# Notebook 01 — Data Loading & Exploration
## Machine Learning for Product Gain/Loss Prediction
### A Case Study of PMS and AGO in Depot Operations
**Author:** Uzoma Nnaemeka Eze | MSc Data Science | University of Europe for Applied Sciences, Potsdam  
**Supervisor:** Prof. Dr. Iftikhar Ahmed  

---
This notebook loads and explores the two operational dataset layers:
- **M2 Summary Reports** — depot-level daily stock records (Jan–Dec 2025, 730 observations)
- **D2 Tank Reports** — individual tank daily dip readings (Feb–Dec 2025, 4,264 tank-day records)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully.")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")


## 1. Load M2 Depot-Level Summary Data

In [ ]:
def load_m2_sheet(path, sheet):
    """
    Parse a product sheet from the APD M2 summary workbook.
    Each row represents one day of depot-level operations.
    Returns a cleaned DataFrame with typed columns.
    """
    df = pd.read_excel(path, sheet_name=sheet, header=None)
    data = df.iloc[2:].copy()
    data.columns = range(len(data.columns))

    result = pd.DataFrame()
    result['Date']             = pd.to_datetime(data[0],  errors='coerce')
    result['Opening_Stock']    = pd.to_numeric(data[1],   errors='coerce')
    result['Total_Receipts']   = pd.to_numeric(data[10],  errors='coerce')
    result['Total_Deliveries'] = pd.to_numeric(data[16],  errors='coerce')
    result['Book_Balance']     = pd.to_numeric(data[17],  errors='coerce')
    result['Closing_Stock']    = pd.to_numeric(data[18],  errors='coerce')
    result['Loss_Gain']        = pd.to_numeric(data[19],  errors='coerce')
    result['LG_Pct']           = pd.to_numeric(data[20],  errors='coerce')

    result = result.dropna(subset=['Date', 'Loss_Gain'])
    result = result[result['Date'].dt.year == 2025].reset_index(drop=True)
    return result

# Update path to your local file location
M2_PATH = 'data/M2_REPORTS-2025-APD_DEPOTS.xlsx'

# For demonstration — load from saved CSVs if available
try:
    ago = pd.read_csv('data/ago_m2.csv', parse_dates=['Date'])
    pms = pd.read_csv('data/pms_m2.csv', parse_dates=['Date'])
    print("Loaded from CSV cache.")
except:
    print("Load from Excel file using load_m2_sheet(M2_PATH, 'APD AGO 2025')")
    ago = pd.DataFrame()
    pms = pd.DataFrame()


## 2. M2 Dataset Overview

In [ ]:
for product, df in [('AGO', ago), ('PMS', pms)]:
    if len(df) == 0:
        print(f"{product}: No data loaded.")
        continue
    print(f"\n{'='*55}")
    print(f"  {product} — M2 Daily Summary")
    print(f"{'='*55}")
    print(f"Records       : {len(df)}")
    print(f"Date range    : {df['Date'].min().date()} to {df['Date'].max().date()}")
    print(f"Loss days     : {(df['Loss_Gain']<0).sum()} ({(df['Loss_Gain']<0).mean()*100:.1f}%)")
    print(f"Gain days     : {(df['Loss_Gain']>0).sum()} ({(df['Loss_Gain']>0).mean()*100:.1f}%)")
    print(f"Zero days     : {(df['Loss_Gain']==0).sum()} ({(df['Loss_Gain']==0).mean()*100:.1f}%)")
    print(f"Missing values: {df.isnull().sum().sum()}")
    print(f"\nLoss/Gain Statistics (litres):")
    print(df['Loss_Gain'].describe().round(2))


## 3. Loss/Gain Distribution Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Loss/Gain Distribution — AGO and PMS (M2 Daily Summary 2025)',
             fontsize=13, fontweight='bold')

for ax, (product, df, color) in zip(axes, [('AGO', ago, '#1A5F9E'), ('PMS', pms, '#C94E2D')]):
    if len(df) == 0:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        continue
    ax.hist(df['Loss_Gain']/1000, bins=40, color=color, alpha=0.8, edgecolor='white')
    ax.axvline(0, color='black', lw=1.5, linestyle='--')
    ax.set_title(f'{product} — Daily Loss/Gain Distribution', fontweight='bold', color=color)
    ax.set_xlabel('Loss/Gain (000 litres)')
    ax.set_ylabel('Frequency')
    ax.grid(True, alpha=0.3)
    ax.set_facecolor('#FAFAFA')

plt.tight_layout()
plt.savefig('reports/fig_loss_gain_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")


## 4. Load D2 Tank-Level Data

In [ ]:
def safe_num(val):
    """Convert cell value to float. NIL/NaN/empty cells return 0.0."""
    try:
        v = str(val).strip().upper()
        if v in ['NIL', 'NAN', 'NONE', '', 'NIT']:
            return 0.0
        return float(val)
    except:
        return np.nan

def parse_day_sheet(df_raw):
    """
    Extract individual tank records from a single daily D2 sheet.
    Tank rows are identified by Tank ID starting with T-.
    Only Natural Temperature section is used (Temperature > 5 degrees).
    """
    records = []
    for i, row in df_raw.iterrows():
        tank    = str(row[0]).strip()
        product = str(row[1]).strip() if pd.notna(row[1]) else ''
        if not tank.startswith('T-') or product not in ['PMS', 'AGO']:
            continue
        records.append({
            'Tank':           tank,
            'Product':        product,
            'Opening_Dip_mm': safe_num(row[2]),
            'Water_Dip_mm':   safe_num(row[3]),
            'Temperature':    safe_num(row[5]),
            'Opening_Vol':    safe_num(row[6]),
            'Receipt':        safe_num(row[7]),
            'Dispatch':       safe_num(row[8]),
            'Closing_Dip_mm': safe_num(row[10]),
            'Loss_Gain':      safe_num(row[15]),
            'LG_Pct':         safe_num(row[16]),
        })
    return records

# Load from saved CSV if available
try:
    tank_df = pd.read_csv('data/tank_clean.csv', parse_dates=['Date'])
    print(f"Tank data loaded: {len(tank_df)} records")
    print(f"Date range: {tank_df['Date'].min().date()} to {tank_df['Date'].max().date()}")
    print(f"Tanks: {sorted(tank_df['Tank'].unique())}")
    print(f"AGO: {(tank_df['Product']=='AGO').sum()} | PMS: {(tank_df['Product']=='PMS').sum()}")
except:
    print("Load D2 Excel files using parse_day_sheet() function above.")
    tank_df = pd.DataFrame()


## 5. Tank-Level Summary Statistics

In [ ]:
if len(tank_df) > 0:
    tank_df['Loss_Flag'] = (tank_df['Loss_Gain'] < 0).astype(int)

    print("Tank-Level Summary by Product:")
    print("="*55)
    for product in ['AGO', 'PMS']:
        sub = tank_df[tank_df['Product']==product]
        print(f"\n{product}:")
        print(f"  Records     : {len(sub)}")
        print(f"  Tanks       : {sorted(sub['Tank'].unique())}")
        print(f"  Loss events : {sub['Loss_Flag'].sum()} ({sub['Loss_Flag'].mean()*100:.1f}%)")
        print(f"  Dip range   : {sub['Opening_Dip_mm'].min():.0f} mm to {sub['Opening_Dip_mm'].max():.0f} mm")
        print(f"  Water dip   : {(sub['Water_Dip_mm']>0).sum()} tank-days with water detected")
